In [ ]:
# ================================================================
# BirdCLEF+ 2026 — COMPLETE PIPELINE
# EfficientNet-B2 (noisy-student) | Log-Mel Spectrogram
# Weighted BCE | Mixup | SpecAugment | Site-aware CV
# Soundscape Fine-tuning | Sliding-window TTA Inference
# ================================================================
# NOTEBOOK STRUCTURE
#   Cell 1  : Installs
#   Cell 2  : Imports & CFG
#   Cell 3  : Metadata loading + label encoding
#   Cell 4  : Site-aware GroupKFold + class weights
#   Cell 5  : Audio utilities + mel spectrogram
#   Cell 6  : Augmentations
#   Cell 7  : Dataset (clips) + Mixup
#   Cell 8  : Model (EfficientNet-B2 head)
#   Cell 9  : Loss + metrics
#   Cell 10 : Train / val loops
#   Cell 11 : Phase 1 — run_training()
#   Cell 12 : Soundscape dataset + Phase 2 fine-tuning
#   Cell 13 : Phase 3 — inference + submission
# ================================================================


# ================================================================
# CELL 1 — Install dependencies
# ================================================================
# %%
# Run this cell first in a Kaggle notebook
# !pip install timm librosa audioread --quiet


# ================================================================
# CELL 2 — Imports & Config
# ================================================================
# %%
import os, gc, math, random, warnings, time
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

import librosa

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
from torch.cuda.amp import GradScaler, autocast

from tqdm.auto import tqdm


# ── Single config object — change everything here ────────────────
class CFG:

    # ── Paths ────────────────────────────────────────────────────
    COMP_DIR             = Path("/kaggle/input/birdclef-2026")
    TRAIN_AUDIO_DIR      = COMP_DIR / "train_audio"
    TRAIN_SOUNDSCAPE_DIR = COMP_DIR / "train_soundscapes"
    TRAIN_CSV            = COMP_DIR / "train_metadata.csv"
    SOUNDSCAPE_CSV       = COMP_DIR / "train_soundscapes_labels.csv"
    TEST_DIR             = COMP_DIR / "test_soundscapes"
    SAMPLE_SUB           = COMP_DIR / "sample_submission.csv"
    OUTPUT_DIR           = Path("/kaggle/working")

    # ── Audio ────────────────────────────────────────────────────
    SR         = 32_000        # sample rate
    DURATION   = 5             # seconds per crop
    N_SAMPLES  = SR * DURATION # 160 000 samples per crop

    # ── Mel spectrogram ──────────────────────────────────────────
    N_FFT      = 1024
    HOP_LENGTH = 320           # 100 frames/s  →  500 frames per 5 s
    N_MELS     = 128
    FMIN       = 20
    FMAX       = 16_000

    # ── Model ────────────────────────────────────────────────────
    # tf_efficientnet_b2_ns  →  noisy-student pretrained, strong + fast
    MODEL_NAME  = "tf_efficientnet_b2_ns"
    PRETRAINED  = True
    IN_CHANNELS = 1            # single-channel log-mel

    # ── Training ─────────────────────────────────────────────────
    SEED         = 42
    FOLDS        = 5
    TRAIN_FOLDS  = [0, 1, 2, 3, 4]   # set to [0] for a quick smoke test
    EPOCHS       = 30
    BATCH_SIZE   = 32
    NUM_WORKERS  = 2
    LR           = 1e-3
    MIN_LR       = 1e-6
    WEIGHT_DECAY = 1e-2

    # ── Augmentation ─────────────────────────────────────────────
    MIXUP_ALPHA  = 0.5
    MIXUP_PROB   = 0.5

    # ── Inference / TTA ──────────────────────────────────────────
    # Fractional second shifts for test-time augmentation
    TTA_SHIFTS   = [0.0, 0.5, 1.0]

    # ── AMP ──────────────────────────────────────────────────────
    USE_AMP = True

    # ── Debug (set True to run on 500 samples only) ───────────────
    DEBUG   = False
    N_DEBUG = 500


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(CFG.SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


# ================================================================
# CELL 3 — Load metadata, encode labels
# ================================================================
# %%
train_df   = pd.read_csv(CFG.TRAIN_CSV)
sample_sub = pd.read_csv(CFG.SAMPLE_SUB)

# ── Species list derived from sample submission ───────────────────
SPECIES     = [c for c in sample_sub.columns if c not in ("row_id", "filename", "end_time")]
NUM_CLASSES = len(SPECIES)
SPECIES2IDX = {s: i for i, s in enumerate(SPECIES)}
IDX2SPECIES = {i: s for s, i in SPECIES2IDX.items()}

print(f"Classes : {NUM_CLASSES}")
print(f"Train   : {len(train_df)} rows")
print(train_df.head(2))

if CFG.DEBUG:
    train_df = train_df.sample(CFG.N_DEBUG, random_state=CFG.SEED).reset_index(drop=True)

# ── Build filepath ────────────────────────────────────────────────
train_df["filepath"] = train_df["filename"].apply(
    lambda x: str(CFG.TRAIN_AUDIO_DIR / x)
)


def encode_labels(row) -> np.ndarray:
    """
    Convert `primary_label` + `secondary_labels` columns into a
    multi-hot float32 vector of length NUM_CLASSES.
    """
    vec = np.zeros(NUM_CLASSES, dtype=np.float32)

    # Primary label
    primary = row.get("primary_label", "")
    if isinstance(primary, str) and primary in SPECIES2IDX:
        vec[SPECIES2IDX[primary]] = 1.0

    # Secondary labels (stored as "['sp1','sp2']" string or actual list)
    secondary = row.get("secondary_labels", "")
    if pd.notna(secondary) and str(secondary) not in ("", "[]"):
        try:
            parsed = eval(secondary) if isinstance(secondary, str) else secondary
            for sp in parsed:
                sp = sp.strip().strip("'\"")
                if sp in SPECIES2IDX:
                    vec[SPECIES2IDX[sp]] = 1.0
        except Exception:
            pass

    return vec


train_df["label_vec"] = train_df.apply(encode_labels, axis=1)


# ================================================================
# CELL 4 — Site-aware GroupKFold + class weights
# ================================================================
# %%

# ── Groups for site-aware split ───────────────────────────────────
#   Priority: author → site column → lat/lon grid
if "author" in train_df.columns:
    groups = train_df["author"].fillna("unknown")
elif "site" in train_df.columns:
    groups = train_df["site"].fillna("unknown")
else:
    lat = train_df.get("latitude",  pd.Series(np.zeros(len(train_df)))).fillna(0)
    lon = train_df.get("longitude", pd.Series(np.zeros(len(train_df)))).fillna(0)
    groups = (lat // 5).astype(str) + "_" + (lon // 5).astype(str)

gkf = GroupKFold(n_splits=CFG.FOLDS)
train_df["fold"] = -1
for fold, (_, val_idx) in enumerate(gkf.split(train_df, groups=groups)):
    train_df.loc[val_idx, "fold"] = fold

print("Fold distribution:\n", train_df["fold"].value_counts().sort_index())

# ── Class weights ─────────────────────────────────────────────────
#   Rare classes get higher weight → prevents frequent species dominating loss
label_matrix  = np.stack(train_df["label_vec"].values)        # (N, C)
class_counts  = label_matrix.sum(axis=0).clip(min=1)
class_weights = (len(train_df) / (NUM_CLASSES * class_counts)).astype(np.float32)
class_weights = torch.tensor(class_weights).to(DEVICE)

print(f"Class weight range: {class_weights.min():.2f} – {class_weights.max():.2f}")


# ================================================================
# CELL 5 — Audio utilities + mel spectrogram
# ================================================================
# %%

def load_audio(path: str, sr: int = CFG.SR, duration: float = None) -> np.ndarray:
    """Load audio file, resample, mono. Returns float32 waveform."""
    try:
        audio, _ = librosa.load(path, sr=sr, mono=True, duration=duration)
    except Exception as exc:
        print(f"  [WARN] load_audio failed for {path}: {exc}")
        audio = np.zeros(CFG.N_SAMPLES, dtype=np.float32)
    return audio.astype(np.float32)


def random_crop(audio: np.ndarray, n_samples: int = CFG.N_SAMPLES) -> np.ndarray:
    """Random 5-second crop; tiles short clips."""
    if len(audio) < n_samples:
        repeats = math.ceil(n_samples / max(len(audio), 1))
        audio = np.tile(audio, repeats)
    start = random.randint(0, len(audio) - n_samples)
    return audio[start : start + n_samples]


def fixed_crop(audio: np.ndarray, start_sample: int,
               n_samples: int = CFG.N_SAMPLES) -> np.ndarray:
    """Fixed-position crop for inference; zero-pads if needed."""
    end = start_sample + n_samples
    if len(audio) < end:
        audio = np.pad(audio, (0, end - len(audio)))
    return audio[start_sample:end]


def audio_to_melspec(audio: np.ndarray, sr: int = CFG.SR) -> np.ndarray:
    """
    Convert waveform → log-mel spectrogram normalized to [0, 1].
    Output shape: (N_MELS, T)  e.g. (128, 500) for 5 s at 32 kHz.
    """
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_fft=CFG.N_FFT,
        hop_length=CFG.HOP_LENGTH,
        n_mels=CFG.N_MELS,
        fmin=CFG.FMIN,
        fmax=CFG.FMAX,
        power=2.0,
    )
    mel_db = librosa.power_to_db(mel, ref=np.max, top_db=80.0)
    mel_norm = (mel_db + 80.0) / 80.0   # map [-80, 0] → [0, 1]
    return mel_norm.astype(np.float32)


# ================================================================
# CELL 6 — Augmentations
# ================================================================
# %%

# ── Waveform augmentations ────────────────────────────────────────

def time_shift(audio: np.ndarray, max_shift_frac: float = 0.1) -> np.ndarray:
    """Circular shift by a random fraction of the clip."""
    shift = int(random.uniform(0, max_shift_frac) * len(audio))
    return np.roll(audio, shift)


def add_noise(audio: np.ndarray, noise_level: float = 0.005) -> np.ndarray:
    """Additive Gaussian noise."""
    return audio + np.random.randn(len(audio)).astype(np.float32) * noise_level


def random_gain(audio: np.ndarray, low: float = 0.7, high: float = 1.3) -> np.ndarray:
    """Random amplitude scaling."""
    return audio * random.uniform(low, high)


# ── Spectrogram augmentations ─────────────────────────────────────

def spec_augment(
    mel: np.ndarray,
    time_mask_param: int = 80,
    freq_mask_param: int = 30,
    num_masks: int = 2,
) -> np.ndarray:
    """
    SpecAugment: time masking + frequency masking.
    Operates in-place on a copy of the spectrogram.
    """
    mel = mel.copy()
    T = mel.shape[1]
    F = mel.shape[0]

    for _ in range(num_masks):
        # Time mask
        t = random.randint(0, time_mask_param)
        t0 = random.randint(0, max(T - t, 1))
        mel[:, t0 : t0 + t] = 0.0

        # Frequency mask
        f = random.randint(0, freq_mask_param)
        f0 = random.randint(0, max(F - f, 1))
        mel[f0 : f0 + f, :] = 0.0

    return mel


# ================================================================
# CELL 7 — Dataset (clips) + Mixup
# ================================================================
# %%

class BirdClipDataset(Dataset):
    """
    Dataset for isolated XenoCanto / iNaturalist clips.
    mode='train'  →  random crop + full augmentation stack
    mode='valid'  →  random crop, no spec augment
    """

    def __init__(self, df: pd.DataFrame, mode: str = "train"):
        self.df   = df.reset_index(drop=True)
        self.mode = mode

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row   = self.df.iloc[idx]
        audio = load_audio(row["filepath"])

        if self.mode == "train":
            audio = random_crop(audio)
            audio = time_shift(audio)
            if random.random() < 0.3:
                audio = add_noise(audio)
            if random.random() < 0.3:
                audio = random_gain(audio)
        else:
            audio = random_crop(audio)

        mel = audio_to_melspec(audio)

        if self.mode == "train" and random.random() < 0.7:
            mel = spec_augment(mel)

        mel   = torch.tensor(mel).unsqueeze(0)                    # (1, H, W)
        label = torch.tensor(row["label_vec"], dtype=torch.float32)
        return mel, label


# ── Mixup ─────────────────────────────────────────────────────────

def mixup_data(x: torch.Tensor, y: torch.Tensor, alpha: float = CFG.MIXUP_ALPHA):
    """Apply mixup augmentation to a batch."""
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam


def mixup_loss(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


# ================================================================
# CELL 8 — Model
# ================================================================
# %%

class BirdModel(nn.Module):
    """
    EfficientNet-B2 (noisy-student pretrained) with a multilabel
    sigmoid classification head for NUM_CLASSES species.

    The backbone expects (B, 1, H, W) single-channel log-mel input.
    timm's in_chans=1 handles weight adaptation automatically.
    """

    def __init__(
        self,
        model_name:  str  = CFG.MODEL_NAME,
        num_classes: int  = NUM_CLASSES,
        pretrained:  bool = CFG.PRETRAINED,
        drop_rate:   float = 0.3,
    ):
        super().__init__()
        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0,          # remove classifier
            global_pool="avg",
            in_chans=CFG.IN_CHANNELS,
        )
        feat_dim = self.backbone.num_features

        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Dropout(drop_rate),
            nn.Linear(feat_dim, 512),
            nn.GELU(),
            nn.Dropout(drop_rate / 2),
            nn.Linear(512, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats  = self.backbone(x)   # (B, feat_dim)
        logits = self.head(feats)   # (B, num_classes)  — raw logits
        return logits               # sigmoid applied at loss / inference time


# ================================================================
# CELL 9 — Loss + metrics
# ================================================================
# %%

class WeightedBCELoss(nn.Module):
    """
    Binary cross-entropy with per-class weights.
    Rare classes get higher weight, preventing frequent species
    from dominating the macro-AUC objective.
    """

    def __init__(self, class_weights: torch.Tensor = None):
        super().__init__()
        self.class_weights = class_weights  # shape (C,)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        loss = F.binary_cross_entropy_with_logits(
            logits, targets, reduction="none"
        )   # (B, C)
        if self.class_weights is not None:
            loss = loss * self.class_weights.unsqueeze(0)
        return loss.mean()


def compute_macro_auc(
    targets: np.ndarray,
    probs:   np.ndarray,
    species: list = SPECIES,
) -> float:
    """
    Macro ROC-AUC over classes that have at least one positive sample.
    Mirrors the competition's class-skipping evaluation logic.
    """
    aucs = []
    for i in range(len(species)):
        y_true = targets[:, i]
        if y_true.sum() == 0:
            continue        # skip — no positive labels for this class
        try:
            aucs.append(roc_auc_score(y_true, probs[:, i]))
        except Exception:
            pass
    return float(np.mean(aucs)) if aucs else 0.0


# ================================================================
# CELL 10 — Train / val loops
# ================================================================
# %%

def train_one_epoch(
    model:     nn.Module,
    loader:    DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    scaler:    GradScaler,
    scheduler = None,
) -> float:
    model.train()
    total_loss = 0.0
    pbar = tqdm(loader, desc="  train", leave=False)

    for mels, labels in pbar:
        mels   = mels.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        # ── Optional mixup ────────────────────────────────────────
        if random.random() < CFG.MIXUP_PROB:
            mels, y_a, y_b, lam = mixup_data(mels, labels)
            with autocast(enabled=CFG.USE_AMP):
                logits = model(mels)
                loss   = mixup_loss(criterion, logits, y_a, y_b, lam)
        else:
            with autocast(enabled=CFG.USE_AMP):
                logits = model(mels)
                loss   = criterion(logits, labels)

        # ── Backward ──────────────────────────────────────────────
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / len(loader)


@torch.no_grad()
def validate(
    model:     nn.Module,
    loader:    DataLoader,
    criterion: nn.Module,
) -> tuple[float, float]:
    model.eval()
    total_loss = 0.0
    all_probs, all_labels = [], []

    for mels, labels in tqdm(loader, desc="  valid", leave=False):
        mels   = mels.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with autocast(enabled=CFG.USE_AMP):
            logits = model(mels)
            loss   = criterion(logits, labels)

        total_loss += loss.item()
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    all_probs  = np.concatenate(all_probs,  axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    macro_auc  = compute_macro_auc(all_labels, all_probs)
    return total_loss / len(loader), macro_auc


# ================================================================
# CELL 11 — Phase 1: Clip training across all folds
# ================================================================
# %%

def run_training() -> None:
    """
    Phase 1 — Train EfficientNet-B2 on isolated clips.
    Saves best checkpoint per fold to CFG.OUTPUT_DIR.
    Prints per-fold AUC and final CV mean ± std.
    """
    best_aucs = []

    for fold in CFG.TRAIN_FOLDS:
        print(f"\n{'═'*60}")
        print(f"  FOLD {fold}  |  {CFG.MODEL_NAME}")
        print(f"{'═'*60}")

        tr_df = train_df[train_df["fold"] != fold].reset_index(drop=True)
        vl_df = train_df[train_df["fold"] == fold].reset_index(drop=True)

        tr_ds = BirdClipDataset(tr_df, mode="train")
        vl_ds = BirdClipDataset(vl_df, mode="valid")

        tr_loader = DataLoader(
            tr_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,
            num_workers=CFG.NUM_WORKERS, pin_memory=True, drop_last=True,
        )
        vl_loader = DataLoader(
            vl_ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
            num_workers=CFG.NUM_WORKERS, pin_memory=True,
        )

        model     = BirdModel().to(DEVICE)
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY
        )
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=CFG.LR,
            total_steps=len(tr_loader) * CFG.EPOCHS,
            pct_start=0.1,
            anneal_strategy="cos",
            final_div_factor=CFG.LR / CFG.MIN_LR,
        )
        criterion = WeightedBCELoss(class_weights=class_weights)
        scaler    = GradScaler(enabled=CFG.USE_AMP)

        best_auc  = 0.0
        ckpt_path = CFG.OUTPUT_DIR / f"model_fold{fold}.pt"

        for epoch in range(CFG.EPOCHS):
            t0      = time.time()
            tr_loss = train_one_epoch(model, tr_loader, optimizer, criterion, scaler, scheduler)
            vl_loss, vl_auc = validate(model, vl_loader, criterion)
            elapsed = time.time() - t0

            flag = ""
            if vl_auc > best_auc:
                best_auc = vl_auc
                torch.save(model.state_dict(), ckpt_path)
                flag = "  ✓ best"

            print(
                f"  Ep {epoch+1:03d}/{CFG.EPOCHS}"
                f"  tr {tr_loss:.4f}"
                f"  vl {vl_loss:.4f}"
                f"  auc {vl_auc:.4f}"
                f"  {elapsed:.0f}s"
                f"{flag}"
            )

        print(f"\n  Fold {fold} best AUC: {best_auc:.4f}  →  {ckpt_path.name}")
        best_aucs.append(best_auc)

        del model, optimizer, scheduler, scaler
        del tr_loader, vl_loader, tr_ds, vl_ds
        gc.collect()
        torch.cuda.empty_cache()

    print(f"\n{'═'*60}")
    print(f"  CV AUC : {np.mean(best_aucs):.4f} ± {np.std(best_aucs):.4f}")
    print(f"  Folds  : {[f'{a:.4f}' for a in best_aucs]}")
    print(f"{'═'*60}")


# ── RUN PHASE 1 ───────────────────────────────────────────────────
# (comment out when debugging other cells)
run_training()


# ================================================================
# CELL 12 — Phase 2: Soundscape fine-tuning
# ================================================================
# %%

class SoundscapeDataset(Dataset):
    """
    Fine-tuning dataset built from labeled competition soundscapes.
    These are 1-minute recordings; labels are given per 5-second window.
    This dataset yields one (mel, label) pair per labeled window.
    """

    def __init__(
        self,
        df:        pd.DataFrame,
        audio_dir: Path = CFG.TRAIN_SOUNDSCAPE_DIR,
        mode:      str  = "train",
    ):
        self.df        = df.reset_index(drop=True)
        self.audio_dir = Path(audio_dir)
        self.mode      = mode

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row      = self.df.iloc[idx]
        filename = row["filename"]
        end_time = float(row["end_time"])

        audio = load_audio(str(self.audio_dir / filename), sr=CFG.SR)

        # Crop the labeled 5-second window (with a tiny random jitter in train)
        start_sec    = max(end_time - CFG.DURATION, 0.0)
        jitter       = random.uniform(-0.2, 0.2) if self.mode == "train" else 0.0
        start_sample = int((start_sec + jitter) * CFG.SR)
        start_sample = max(start_sample, 0)
        audio        = fixed_crop(audio, start_sample)

        if self.mode == "train":
            if random.random() < 0.2:
                audio = add_noise(audio, noise_level=0.002)
            if random.random() < 0.2:
                audio = random_gain(audio, 0.8, 1.2)

        mel = audio_to_melspec(audio)

        if self.mode == "train" and random.random() < 0.5:
            mel = spec_augment(mel, time_mask_param=40, freq_mask_param=20)

        mel = torch.tensor(mel).unsqueeze(0)   # (1, H, W)

        # Build label vector from "birds" column (space-separated species codes)
        label   = np.zeros(NUM_CLASSES, dtype=np.float32)
        birds   = row.get("birds", "")
        if pd.notna(birds) and str(birds) not in ("", "nocall"):
            for sp in str(birds).split():
                sp = sp.strip()
                if sp in SPECIES2IDX:
                    label[SPECIES2IDX[sp]] = 1.0

        return mel, torch.tensor(label, dtype=torch.float32)


def run_soundscape_finetuning(
    fold:      int   = 0,
    ft_epochs: int   = 5,
    ft_lr:     float = 1e-4,
) -> None:
    """
    Phase 2 — Fine-tune on labeled soundscapes.

    Strategy:
      Epochs 0-1 : backbone frozen, only head trains (fast domain adaptation)
      Epoch  2+  : backbone unfreezes at 10× lower LR (discriminative LR)

    Saves fine-tuned checkpoint as model_fold{fold}_ft.pt
    """
    if not CFG.SOUNDSCAPE_CSV.exists():
        print("  No soundscape labels CSV found — skipping fine-tuning.")
        return

    sc_df  = pd.read_csv(CFG.SOUNDSCAPE_CSV)
    sc_ds  = SoundscapeDataset(sc_df, mode="train")
    sc_ldr = DataLoader(
        sc_ds, batch_size=16, shuffle=True,
        num_workers=CFG.NUM_WORKERS, pin_memory=True, drop_last=True,
    )

    ckpt_path = CFG.OUTPUT_DIR / f"model_fold{fold}.pt"
    ft_path   = CFG.OUTPUT_DIR / f"model_fold{fold}_ft.pt"

    if not ckpt_path.exists():
        print(f"  Checkpoint {ckpt_path.name} not found — run training first.")
        return

    print(f"\n{'═'*60}")
    print(f"  SOUNDSCAPE FINE-TUNING  fold={fold}  epochs={ft_epochs}")
    print(f"{'═'*60}")

    model = BirdModel().to(DEVICE)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))

    # Phase A: freeze backbone
    for p in model.backbone.parameters():
        p.requires_grad = False

    optimizer = torch.optim.AdamW(
        model.head.parameters(), lr=ft_lr, weight_decay=CFG.WEIGHT_DECAY
    )
    criterion = WeightedBCELoss(class_weights=class_weights)
    scaler    = GradScaler(enabled=CFG.USE_AMP)

    for epoch in range(ft_epochs):
        # Phase B: unfreeze backbone at epoch 2 with discriminative LR
        if epoch == 2:
            for p in model.backbone.parameters():
                p.requires_grad = True
            optimizer = torch.optim.AdamW(
                [
                    {"params": model.backbone.parameters(), "lr": ft_lr * 0.1},
                    {"params": model.head.parameters(),     "lr": ft_lr},
                ],
                weight_decay=CFG.WEIGHT_DECAY,
            )

        tr_loss = train_one_epoch(model, sc_ldr, optimizer, criterion, scaler)
        print(f"  FT Ep {epoch+1}/{ft_epochs}  |  loss {tr_loss:.4f}")

    torch.save(model.state_dict(), ft_path)
    print(f"\n  Fine-tuned model → {ft_path.name}")
    del model; gc.collect(); torch.cuda.empty_cache()


# ── RUN PHASE 2 for every fold ────────────────────────────────────
for _fold in CFG.TRAIN_FOLDS:
    run_soundscape_finetuning(fold=_fold, ft_epochs=5, ft_lr=1e-4)


# ================================================================
# CELL 13 — Phase 3: Inference + submission
# ================================================================
# %%

@torch.no_grad()
def predict_soundscape(
    model:      nn.Module,
    audio_path: Path,
    tta_shifts: list = CFG.TTA_SHIFTS,
) -> dict[int, np.ndarray]:
    """
    Slide a 5-second window over a 1-minute soundscape.
    For each 5 s segment, apply TTA by shifting the crop origin slightly.

    Returns:
        {end_time (int, 5…60): prob_vector (np.ndarray, shape NUM_CLASSES)}
    """
    CLIP_SECS     = 60
    total_samples = CFG.SR * CLIP_SECS

    audio = load_audio(str(audio_path), sr=CFG.SR)
    # Normalise length to exactly 60 s
    if len(audio) < total_samples:
        audio = np.pad(audio, (0, total_samples - len(audio)))
    else:
        audio = audio[:total_samples]

    model.eval()
    results: dict[int, np.ndarray] = {}

    for end_time in range(5, 65, 5):   # 5, 10, ..., 60
        start_sec  = end_time - CFG.DURATION
        tta_probs  = []

        for shift in tta_shifts:
            s_start = int((start_sec + shift) * CFG.SR)
            s_start = max(0, min(s_start, total_samples - CFG.N_SAMPLES))
            crop    = audio[s_start : s_start + CFG.N_SAMPLES]
            if len(crop) < CFG.N_SAMPLES:
                crop = np.pad(crop, (0, CFG.N_SAMPLES - len(crop)))

            mel   = audio_to_melspec(crop)
            mel_t = torch.tensor(mel).unsqueeze(0).unsqueeze(0).to(DEVICE)  # (1,1,H,W)

            logits = model(mel_t)
            probs  = torch.sigmoid(logits).cpu().numpy()[0]
            tta_probs.append(probs)

        results[end_time] = np.mean(tta_probs, axis=0)

    return results


def run_inference() -> pd.DataFrame:
    """
    Phase 3 — Generate submission.csv.

    Loads every fold model (prefers *_ft.pt over *.pt), runs
    sliding-window + TTA inference, and averages probabilities
    across folds (ensemble).
    """
    # ── Collect test files ────────────────────────────────────────
    test_files = sorted(CFG.TEST_DIR.glob("*.ogg"))
    if not test_files:
        test_files = sorted(CFG.TEST_DIR.glob("*.wav"))
    print(f"Test soundscapes: {len(test_files)}")

    # ── Load fold models ──────────────────────────────────────────
    models = []
    for fold in CFG.TRAIN_FOLDS:
        ft_path  = CFG.OUTPUT_DIR / f"model_fold{fold}_ft.pt"
        std_path = CFG.OUTPUT_DIR / f"model_fold{fold}.pt"
        path     = ft_path if ft_path.exists() else std_path
        if not path.exists():
            print(f"  [WARN] No checkpoint for fold {fold} — skipping.")
            continue
        m = BirdModel().to(DEVICE)
        m.load_state_dict(torch.load(path, map_location=DEVICE))
        m.eval()
        models.append(m)
        print(f"  Loaded {path.name}")

    if not models:
        raise RuntimeError("No checkpoints found! Run training first.")

    # ── Inference loop ────────────────────────────────────────────
    rows = []
    for audio_path in tqdm(test_files, desc="Inference"):
        filename = audio_path.name

        # Collect per-fold predictions
        fold_preds = [predict_soundscape(m, audio_path) for m in models]

        # Build one submission row per 5-second window
        for end_time in range(5, 65, 5):
            probs = np.mean([fp[end_time] for fp in fold_preds], axis=0)
            row   = {
                "row_id"   : f"{filename}_{end_time}",
                "filename" : filename,
                "end_time" : end_time,
            }
            for i, sp in enumerate(SPECIES):
                row[sp] = float(probs[i])
            rows.append(row)

    # ── Save submission ───────────────────────────────────────────
    sub      = pd.DataFrame(rows)
    sub_path = CFG.OUTPUT_DIR / "submission.csv"
    sub.to_csv(sub_path, index=False)

    print(f"\nSubmission saved → {sub_path}")
    print(f"Shape  : {sub.shape}")
    print(sub.head(3))
    return sub


# ── RUN PHASE 3 ───────────────────────────────────────────────────
submission = run_inference()


# ================================================================
# DONE
# ================================================================
# submission.csv is in /kaggle/working — submit directly.
#
# Quick checklist before submitting:
#   [ ] Verify NUM_CLASSES matches sample_submission column count
#   [ ] Verify test_files are found (check TEST_DIR path)
#   [ ] Confirm inference runs within 90-minute CPU budget
#       (reduce TTA_SHIFTS or TRAIN_FOLDS if too slow on CPU)
#   [ ] Check sub.isnull().sum().sum() == 0
# ================================================================